In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import chess
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from tqdm import tqdm

In [ ]:
# Standard chess games
from datasets import load_dataset
ds = load_dataset("Lichess/standard-chess-games")

In [ ]:
# Classical advanced games
# Web download dataset from Hugging Face Hub
file_path = hf_hub_download(repo_id="angeluriot/chess_games", filename="dataset.parquet", repo_type="dataset")

# Load dataset
df = pd.read_parquet(file_path, columns=["moves_uci", "winner", "end_type"])

# Shuffle (recommended so splits are not biased)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into 100 parts
splits = np.array_split(df, 100)

# Save each split
for i, split_df in enumerate(splits):
    output_path = f"data/chess_dataset_part_{i}.parquet"
    split_df.to_parquet(output_path, index=False)
    print(f"Saved {output_path} with {len(split_df)} rows")

del file_path, df, splits, split_df, output_path

In [ ]:
# Load data
df = pd.read_parquet("data/chess_dataset_part_0.parquet")

In [ ]:
def moves_to_boards(moves: list[str], sample_every: int = 3) -> list[chess.Board]:
    """Replay moves and sample board states."""
    board = chess.Board()
    boards = []
    for i, move in enumerate(moves):
        try:
            board.push_san(move)
        except Exception:
            break  # corrupted game, stop early
        if i % sample_every == 0:
            boards.append(board.copy())
    return boards

def board_to_features(board: chess.Board) -> np.ndarray:
    features = np.zeros(787, dtype=np.float32)

    piece_order = [
        (chess.KING,   chess.WHITE),
        (chess.QUEEN,  chess.WHITE),
        (chess.ROOK,   chess.WHITE),
        (chess.BISHOP, chess.WHITE),
        (chess.KNIGHT, chess.WHITE),
        (chess.PAWN,   chess.WHITE),
        (chess.KING,   chess.BLACK),
        (chess.QUEEN,  chess.BLACK),
        (chess.ROOK,   chess.BLACK),
        (chess.BISHOP, chess.BLACK),
        (chess.KNIGHT, chess.BLACK),
        (chess.PAWN,   chess.BLACK),
    ]
    for i, (piece_type, color) in enumerate(piece_order):
        for square in board.pieces(piece_type, color):
            features[i * 64 + square] = 1.0

    features[768] = float(board.has_kingside_castling_rights(chess.WHITE))
    features[769] = float(board.has_queenside_castling_rights(chess.WHITE))
    features[770] = float(board.has_kingside_castling_rights(chess.BLACK))
    features[771] = float(board.has_queenside_castling_rights(chess.BLACK))

    if board.ep_square is not None:
        features[772 + chess.square_file(board.ep_square)] = 1.0

    features[780] = float(board.turn == chess.WHITE)

    halfmove = min(board.halfmove_clock, 15)
    for bit in range(4):
        features[781 + bit] = float((halfmove >> bit) & 1)

    features[785] = float(board.is_repetition(2))
    features[786] = float(board.is_repetition(3))

    return features

class ChessDataset(Dataset):
    def __init__(self, df: pd.DataFrame, sample_every: int = 3):
        self.samples = []  # list of (features, label)

        for _, row in df.iterrows():
            moves  = row["moves_uci"] 
            winner = row["winner"]         
            label  = 0 if winner == "black" else 2 if winner == "white" else 1

            boards = moves_to_boards(moves, sample_every)
            for board in boards:
                features = board_to_features(board)
                self.samples.append((features, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        features, label = self.samples[idx]
        return torch.tensor(features), torch.tensor(label, dtype=torch.long)

In [ ]:
dataset = ChessDataset(df, sample_every=3)

In [2]:
from model import ChessResNet

model = ChessResNet()

In [ ]:
val_split = 0.05
batch_size = 128
lr = 1e-3
epochs = 10

val_size   = int(len(dataset) * val_split)
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)


In [ ]:

best_val_loss = float("inf")

for epoch in range(epochs):
    # --- Train ---
    model.train()
    train_loss = 0.0
    with tqdm(total=len(train_loader), desc=f"Train {epoch+1}/{epochs}", leave=False, unit="batch") as train_bar:
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(features)
            loss   = torch.nn.functional.cross_entropy(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            train_bar.update(1)

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    correct  = 0
    total    = 0
    with tqdm(total=len(val_loader), desc=f"Val {epoch+1}/{epochs}", leave=False, unit="batch") as val_bar:
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                logits = model(features)
                val_loss += torch.nn.functional.cross_entropy(logits, labels).item()
                correct  += (logits.argmax(dim=1) == labels).sum().item()
                total    += labels.size(0)
                val_bar.update(1)

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    accuracy    = correct / total
    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {accuracy:.3f}")

    scheduler.step()

    # --- Checkpoint ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  → checkpoint saved (val_loss={val_loss:.4f})")

In [ ]:

class Bot4:
    def __init__(self, model):
        super().__init__()
        self.board = chess.Board()
        self.model = model

    def get_feature_vector(self):
        board = self.board
        features = np.zeros(787, dtype=np.float32)

        # --- Piece planes [0:768] ---
        piece_order = [
            (chess.KING,   chess.WHITE),   # [0:64]
            (chess.QUEEN,  chess.WHITE),   # [64:128]
            (chess.ROOK,   chess.WHITE),   # [128:192]
            (chess.BISHOP, chess.WHITE),   # [192:256]
            (chess.KNIGHT, chess.WHITE),   # [256:320]
            (chess.PAWN,   chess.WHITE),   # [320:384]
            (chess.KING,   chess.BLACK),   # [384:448]
            (chess.QUEEN,  chess.BLACK),   # [448:512]
            (chess.ROOK,   chess.BLACK),   # [512:576]
            (chess.BISHOP, chess.BLACK),   # [576:640]
            (chess.KNIGHT, chess.BLACK),   # [640:704]
            (chess.PAWN,   chess.BLACK),   # [704:768]
        ]

        for i, (piece_type, color) in enumerate(piece_order):
            for square in board.pieces(piece_type, color):
                features[i * 64 + square] = 1.0

        # --- Castling rights [768:772] ---
        features[768] = float(board.has_kingside_castling_rights(chess.WHITE))
        features[769] = float(board.has_queenside_castling_rights(chess.WHITE))
        features[770] = float(board.has_kingside_castling_rights(chess.BLACK))
        features[771] = float(board.has_queenside_castling_rights(chess.BLACK))

        # --- En passant [772:780] ---
        if board.ep_square is not None:
            ep_file = chess.square_file(board.ep_square)  # 0-7
            features[772 + ep_file] = 1.0

        # --- Side to move [780] ---
        features[780] = float(board.turn == chess.WHITE)

        # --- Halfmove clock [781:785] ---
        # Encode as 4 bits (supports up to 100 moves for 50-move rule)
        halfmove = min(board.halfmove_clock, 15)  # cap at 15 for 4 bits
        for bit in range(4):
            features[781 + bit] = float((halfmove >> bit) & 1)

        # --- Repetition count [785:787] ---
        features[785] = float(board.is_repetition(2))
        features[786] = float(board.is_repetition(3))

        return torch.tensor(features).unsqueeze(0) # (1, 787)

    def position_evaluation(self):
        board = self.board

        if board.is_game_over():
            winner = chess.WHITE if board.result() == '1-0' else chess.BLACK if board.result() == '0-1' else None
            if winner == chess.WHITE:
                score = float('inf')
            elif winner == chess.BLACK:
                score = float('-inf')
            else:
                score = 0
            return score

        feature = self.get_feature_vector()
        score = self.model.predict_value(feature).item()
        return score

    def minimax(self, depth, alpha, beta, maximizing_player):
        board = self.board

        if depth == 0 or board.is_game_over():
            return self.position_evaluation()

        if maximizing_player:
            value = float("-inf")
            for move in list(board.legal_moves):
                board.push(move)
                new_value = self.minimax(depth - 1, alpha, beta, False)
                board.pop()

                value = max(value, new_value)
                alpha = max(alpha, value)

                if beta <= alpha:
                    break 
        else:
            value = float("inf")
            for move in list(board.legal_moves):
                board.push(move)
                new_value = self.minimax(depth - 1, alpha, beta, True)
                board.pop()

                value = min(value, new_value)
                beta = min(beta, value)
                if beta <= alpha:
                    break

        return value
        
    def select_move(self):
        depth = 1
        best_move = None
        board = self.board

        if board.turn == chess.WHITE:
            best_value = float("-inf")
            for move in list(board.legal_moves):
                board.push(move)
                value = self.minimax(depth - 1, float("-inf"), float("inf"), False)
                board.pop()
                if value > best_value:
                    best_value = value
                    best_move = move
        else:
            best_value = float("inf")
            for move in list(board.legal_moves):
                board.push(move)
                value = self.minimax(depth - 1, float("-inf"), float("inf"), True)
                board.pop()
                if value < best_value:
                    best_value = value
                    best_move = move

        if best_move is None:
            # select a random move if no best move found
            best_move = list(board.legal_moves)[0]

        return best_move

In [47]:
model = ChessResNet()
model.load_state_dict(torch.load("best_model.pt"))
bot = Bot4(model)

In [ ]:
board = chess.Board()

while not board.is_game_over():

    move = bot.select_move()
    print(f"Bot selects: {move}")
    board.push(move)
    bot.board.push(move)

    # input move from user
    user_move = input("Your move (in UCI format, e.g. e2e4): ")
    try:
        board.push_uci(user_move)
        bot.board.push_uci(user_move)
    except ValueError:
        print("Invalid move format. Please enter in UCI format (e.g. e2e4).")
        continue

Bot selects: g2g4
Bot selects: g1h3
Bot selects: c2c4
Bot selects: b1c3
Bot selects: d1c2
Invalid move format. Please enter in UCI format (e.g. e2e4).
Bot selects: c8g4
